In [1]:
# 10_longitudinal_weights.ipynb
# Robustness: longitudinal survey weights. The original-sample longitudinal
# individual weight (I_WGL), available from the second wave onward, is applied as a
# person-constant weight (each member's earliest available longitudinal weight) to
# the original-sample panel. We re-estimate the overall shock effect and the
# insurance buffer (Callaway-Sant'Anna) with and without the longitudinal weight.
# Figure is grayscale, no titles, 600-dpi PNG and PDF; tables to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pyreadstat, seaborn as sns, matplotlib.pyplot as plt
from differences import ATTgt

RAW, PROC, FIG, TAB = "../data/raw", "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# ---- person-constant longitudinal weight: earliest available I_WGL (>0) ----
WL = [(2020, "b"), (2021, "c"), (2022, "d"), (2023, "e"), (2024, "f")]
fr = []
for yr, pf in WL:
    d, _ = pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pf}_ind.sas7bdat", usecols=["PIDWON", "I_WGL"])
    d["I_WGL"] = pd.to_numeric(d["I_WGL"], errors="coerce"); d["year"] = yr
    fr.append(d[d["I_WGL"] > 0])
wl = pd.concat(fr).sort_values("year").groupby("PIDWON")["I_WGL"].first().rename("long_w").reset_index()

p = pd.read_parquet(f"{PROC}/khp_panel.parquet").merge(wl, on="PIDWON", how="inner")  # original-sample panel
for th in ("che25", "che40"):
    p[th + "f"] = p[th].astype(float)
print("Original-sample panel: %d person-years, %d persons" % (len(p), p["PIDWON"].nunique()))

def cs(outcome, weighted=False, split=None):
    cols = ["PIDWON", "year", "cohort", outcome, "long_w"] + (["base_phi"] if split else [])
    d = p[cols].dropna(subset=[outcome]).copy(); d["cohort"] = d["cohort"].replace(0, np.nan)
    d = d.set_index(["PIDWON", "year"])
    a = ATTgt(data=d, cohort_column="cohort", base_period="universal")
    a.fit(formula=outcome, est_method="dr", control_group="not_yet_treated",
          weights_column=("long_w" if weighted else None), sample_split_column=split, progress_bar=False)
    return a

# ---- overall ATT and buffer: unweighted vs longitudinally weighted ----
orows, brows = [], []
for th, lab in [("che25f", "CHE 25%"), ("che40f", "CHE 40%")]:
    u = cs(th, False).aggregate("simple"); u.columns = [str(c) for c in u.columns]
    w = cs(th, True).aggregate("simple"); w.columns = [str(c) for c in w.columns]
    orows.append({"outcome": lab, "unw_att": float(u.iloc[0, 0]), "unw_se": float(u.iloc[0, 1]),
                  "wtd_att": float(w.iloc[0, 0]), "wtd_se": float(w.iloc[0, 1])})
    for wtd in (False, True):
        s = cs(th, wtd, split="base_phi").aggregate("simple"); s.columns = [str(c) for c in s.columns]
        brows.append({"outcome": lab, "weighted": wtd, "att_nophi": float(s.iloc[0, 0]),
                      "att_phi": float(s.iloc[1, 0]), "buffer": float(s.iloc[1, 0]) - float(s.iloc[0, 0])})
overall = pd.DataFrame(orows); overall.to_csv(f"{TAB}/t10_weighted_overall.csv", index=False)
buf = pd.DataFrame(brows); buf.to_csv(f"{TAB}/t10_weighted_buffer.csv", index=False)

# ---- figure: overall ATT and buffer, unweighted vs weighted ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
x = np.arange(2); w_ = 0.36
ax[0].bar(x - w_/2, [100*o["unw_att"] for o in orows], w_, yerr=[100*1.96*o["unw_se"] for o in orows],
          color="0.3", edgecolor="0.3", capsize=4, label="Unweighted")
ax[0].bar(x + w_/2, [100*o["wtd_att"] for o in orows], w_, yerr=[100*1.96*o["wtd_se"] for o in orows],
          color="0.72", edgecolor="0.3", capsize=4, label="Longitudinally weighted")
ax[0].set_xticks(x); ax[0].set_xticklabels(["CHE 25%", "CHE 40%"]); ax[0].set_ylabel("Overall ATT (pp)")
ax[0].legend(frameon=False, fontsize=9)
bd = {(r["outcome"], r["weighted"]): r["buffer"] for r in brows}
ax[1].bar(x - w_/2, [100*bd[(l, False)] for l in ["CHE 25%", "CHE 40%"]], w_, color="0.3", edgecolor="0.3", label="Unweighted")
ax[1].bar(x + w_/2, [100*bd[(l, True)] for l in ["CHE 25%", "CHE 40%"]], w_, color="0.72", edgecolor="0.3", label="Longitudinally weighted")
ax[1].set_xticks(x); ax[1].set_xticklabels(["CHE 25%", "CHE 40%"]); ax[1].set_ylabel("Insurance buffer (pp)")
ax[1].legend(frameon=False, fontsize=9)
save_fig(fig, "fig13_longitudinal_weights")

print("Overall ATT (unweighted vs longitudinally weighted):")
print(overall.round(4).to_string(index=False))
print("\nBuffer (unweighted vs weighted):")
print(buf.round(4).to_string(index=False))


Original-sample panel: 79355 person-years, 14953 persons


Overall ATT (unweighted vs longitudinally weighted):
outcome  unw_att  unw_se  wtd_att  wtd_se
CHE 25%   0.0721  0.0043   0.0505  0.0042
CHE 40%   0.0373  0.0029   0.0242  0.0031

Buffer (unweighted vs weighted):
outcome  weighted  att_nophi  att_phi  buffer
CHE 25%     False     0.1271   0.0566 -0.0705
CHE 25%      True     0.1257   0.0381 -0.0876
CHE 40%     False     0.0745   0.0267 -0.0477
CHE 40%      True     0.0713   0.0162 -0.0551
